# Regime-Switching / Hidden Markov Models -- BTC Direction

Replaces the previous from-scratch attempt in this file (deleted). Every model tried so far
this session -- Logistic Regression, Random Forest, Gradient Boosting, LSTM, CNN-LSTM, across
three feature sets -- has failed to beat Always-up, several significantly worse. That pattern
across five architecturally different classifiers points at the data/label, not the model
class. This file tests a genuinely different *paradigm* instead of a sixth classifier:
model the market as switching between a small number of latent (unobserved) regimes -- e.g.
"bull," "bear," "choppy" -- inferred from observable data, and predict direction from which
regime we're currently in.

**Why this is a different kind of test, not just another model:** every classifier so far
was asked "given today's feature vector, predict tomorrow's direction" directly. A
regime-switching model instead asks two questions in sequence: "which hidden state is the
market in right now?" and "historically, what does that state imply about tomorrow?" It's a
generative model of the *process*, not a discriminative mapping from features to a label --
closer to how the original research the "cycle_state" features in `btc-direcitonal.ipynb`
gestured at, but done properly with an actual latent-state model instead of hand-built phase
buckets.

**Standing methodology, carried over unchanged:** 8-fold expanding chronological
walk-forward, purge at fold boundaries, Always-up baseline only (no Persistence -- removed
project-wide), McNemar's test + fold-clustered bootstrap CI. What's different is what
happens *inside* each fold: instead of `.fit()` on a supervised classifier, a regime model
is fit unsupervised on the training window, then used to infer the current regime for each
test-day prediction.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import binomtest
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

np.random.seed(42)
pd.set_option("display.max_columns", 50)

EXT_DIR = Path("../data/mk4/extended")
MK4_DIR = Path("../data/mk4")
print("Imports OK.")


Imports OK.


## 1. Load Raw Sources

Same raw pulls as before (`btc_ohlc.csv`, `coinmetrics_onchain.csv`,
`blockchain_hashrate_difficulty.csv`, `fee_reward_ratio.csv`, `market_data.csv`,
`economic_data.csv`) -- OHLCV price, on-chain flow, miner economics, macro. Merged onto a
daily calendar backbone with the same weekday forward-fill and economic-release publication
lag as the deleted attempt, since those are correctness fixes, not modeling choices.

In [2]:
btc = pd.read_csv(EXT_DIR / "btc_ohlc.csv", parse_dates=["Date"]).rename(columns={
    "Date": "date", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})

onchain = pd.read_csv(EXT_DIR / "coinmetrics_onchain.csv")
onchain["date"] = pd.to_datetime(onchain["date"]).dt.tz_localize(None)

hashdiff = pd.read_csv(EXT_DIR / "blockchain_hashrate_difficulty.csv", parse_dates=["date"])
fee_reward = pd.read_csv(EXT_DIR / "fee_reward_ratio.csv", parse_dates=["date"])

market = pd.read_csv(EXT_DIR / "market_data.csv", parse_dates=["Date"]).rename(columns={"Date": "date"})
econ = pd.read_csv(EXT_DIR / "economic_data.csv", parse_dates=["Date"]).rename(columns={"Date": "date"})

# Daily calendar backbone = every date BTC has a price for.
calendar = btc[["date"]].copy()

# Market data (weekdays-only): reindex onto the full daily calendar and forward-fill --
# Friday's close carries through the weekend. Backward-looking only.
market_daily = calendar.merge(market, on="date", how="left").sort_values("date")
market_cols = [c for c in market.columns if c != "date"]
market_daily[market_cols] = market_daily[market_cols].ffill()

# Economic releases: forward-fill onto the daily calendar, THEN apply a publication lag --
# CPI/GDP/M2/Unemployment are dated to their reference period, not their release date, so
# using them on their reference date would leak information not yet public at the time.
# Fed Funds Rate (FOMC decisions) and 10Y Treasury Yield (market-observed) need no lag.
econ_daily = calendar.merge(econ, on="date", how="left").sort_values("date")
econ_cols = [c for c in econ.columns if c != "date"]
econ_daily[econ_cols] = econ_daily[econ_cols].ffill()

PUBLICATION_LAG_DAYS = {"CPI": 30, "GDP": 45, "M2 Money Supply": 30, "Unemployment Rate": 30,
                        "Fed Funds Rate": 0, "10Y Treasury Yield": 0}
for col, lag in PUBLICATION_LAG_DAYS.items():
    if lag > 0:
        econ_daily[col] = econ_daily[col].shift(lag)

raw = (btc.merge(onchain, on="date", how="left")
          .merge(hashdiff, on="date", how="left")
          .merge(fee_reward, on="date", how="left")
          .merge(market_daily, on="date", how="left")
          .merge(econ_daily, on="date", how="left"))
raw = raw.sort_values("date").reset_index(drop=True)

print(f"Raw merged table: {raw.shape}, {raw['date'].min().date()} to {raw['date'].max().date()}")
raw.tail(5)


Raw merged table: (4311, 32), 2014-09-17 to 2026-07-06


,date,open,high,low,close,volume,mvrv_ratio,active_addresses,exchange_inflow_btc,exchange_outflow_btc,issuance_usd,exchange_netflow_btc,hashrate,mining_difficulty,fee_reward_ratio,SP500,Nasdaq,Dow,Gold,Silver,Copper,Oil,Natural Gas,Dollar Index,EURUSD,USDCNY,CPI,Fed Funds Rate,10Y Treasury Yield,GDP,M2 Money Supply,Unemployment Rate
4306,2026-07-02,60004.769531,62117.875000,59532.324219,61485.300781,40109297349,1.158186,702851,29809.788317,29364.016090,2.515126e+07,445.772227,8.717666e+08,1.338699e+14,0.009565,7483.240234,25832.669922,52900.070312,4112.700195,60.643002,6.1145,68.690002,3.196,100.860001,1.137799,6.7942,332.407,3.63,4.48,24174.527,22804.5,4.3
4307,2026-07-03,61492.066406,62879.007812,61176.722656,62544.199219,26131813598,1.179299,698135,19840.196464,20622.004572,2.952268e+07,-781.808109,1.004861e+09,1.338699e+14,0.006987,7483.240234,25832.669922,52900.070312,4112.700195,60.643002,6.1145,68.690002,3.196,100.860001,1.142270,6.7886,332.407,3.63,4.49,24174.527,22804.5,4.3
4308,2026-07-04,62545.128906,63398.410156,62287.585938,63088.304688,18608397613,1.189157,550009,9892.373229,8872.396351,3.036226e+07,1019.976879,1.024825e+09,1.338699e+14,0.004940,7483.240234,25832.669922,52900.070312,4112.700195,60.643002,6.1145,68.690002,3.196,100.860001,1.142270,6.7886,332.407,3.63,4.49,24174.527,22804.5,4.3
4309,2026-07-05,63089.058594,63935.847656,62413.992188,63547.882812,18267466154,1.199065,501335,13149.211025,8918.299166,2.544795e+07,4230.911859,8.518024e+08,1.338699e+14,0.005351,7483.240234,25832.669922,52900.070312,4112.700195,60.643002,6.1145,68.690002,3.196,100.860001,1.142270,6.7886,332.407,3.63,4.49,24174.527,22804.5,4.3
4310,2026-07-06,63551.015625,64597.570312,61275.828125,63995.015625,36552222303,1.207395,613733,23639.460203,28675.499590,2.322286e+07,-5036.039387,7.719460e+08,1.338699e+14,0.009995,7537.430176,26121.160156,53055.910156,4155.100098,61.919998,6.1780,68.550003,3.245,100.849998,1.143772,6.7886,332.407,3.63,4.49,24174.527,22804.5,4.3


## 2. Regime Model Design -- The Options

This section is documentation, not code: the choices that shape everything downstream,
made explicit before writing the model itself.

**A. Which implementation.**
- **`statsmodels.tsa.regime_switching.MarkovRegression` / `MarkovAutoregression`** (already
  installed, no new dependency) -- the classic Hamilton (1989) regime-switching model,
  purpose-built for exactly this: a return series whose mean and variance depend on an
  unobserved Markov state, optionally with autoregressive terms and exogenous regressors
  that also switch by regime. This is the standard econometric tool for "is the market in a
  different volatility/drift regime right now."
- **`hmmlearn.GaussianHMM` / `GMMHMM`** (not installed, pure pip install, no native-library
  issues like XGBoost had) -- a more general-purpose HMM library from the ML side rather
  than econometrics. Handles multivariate Gaussian emissions naturally (feed it several
  features per day, not just returns), at the cost of being a less finance-specific tool
  with fewer built-in diagnostics for regime interpretation.
- **Recommendation: start with `statsmodels`** -- it's already available, it's the more
  standard tool for this exact problem, and its regime-conditional mean/variance output
  maps directly onto "predict up if this regime's historical mean return is positive."
  `hmmlearn` is worth a second pass if the multivariate emissions (feeding it more than
  returns/volatility) turn out to matter.

**B. What feeds the model (the observation/emission variables).**
- **Univariate (returns only)** -- the textbook Hamilton setup: regimes differ by the mean
  and variance of daily returns alone. Simplest, most interpretable, least prone to
  overfitting with limited data.
- **Bivariate (returns + volatility)** -- lets a regime be defined by *both* drift and
  choppiness (e.g., "low-vol grind up" vs "high-vol chop" vs "high-vol selloff"), which is
  closer to how "regime" is usually meant informally.
- **Multivariate (returns + on-chain flow + macro)** -- richer, but Gaussian HMMs estimate a
  full covariance matrix per state; with only a handful of states and ~450-3700 rows per
  fold, more than 2-4 observation variables risks the same overfitting-via-more-candidates
  problem that hurt the previous from-scratch attempt, just moved into the emission
  distribution instead of the feature-selection step.
- **Recommendation: start bivariate** (return + a volatility measure), which is the
  smallest step past the textbook univariate case that still lets "choppy" be distinct from
  "trending," and add a third variable only if that shows something worth chasing.

**C. Number of hidden states.**
- **2 states** -- bull/bear. Fewest parameters, most robust to estimate, but forces every
  day into one of two buckets even when the market is genuinely directionless.
- **3 states** -- bull/bear/choppy (or equivalently low-vol/high-vol-up/high-vol-down).
  Matches how this market is usually talked about; more parameters to estimate per fold.
- **4+ states** -- rapidly increases parameter count relative to fold size; a candidate for
  overfitting the same way "more candidate features" was in the prior attempt.
- **Recommendation: test 2 and 3 states as separate model variants**, evaluated
  side by side in the significance table -- treat state count as an honest hyperparameter
  comparison, not a single arbitrary choice.

**D. How a regime becomes a prediction.**
- **Regime's historical mean-return sign** -- if training data assigns regime *k* a mean
  return > 0, predict "up" whenever the model infers the market is in regime *k* today.
  Simplest, directly tied to what the model estimates.
- **Regime's empirical up-day frequency** -- separately count, in training data, what
  fraction of days in each regime were followed by an up day, and threshold at 50%. Slightly
  different from the mean-return version when the return distribution is skewed.
- **Regime as a feature, not the predictor** -- one-hot/probability-of-each-regime fed into
  one of the existing classifiers (Logistic Regression, etc.) as an additional input, rather
  than using the regime model's own output directly. This blurs the "different paradigm" test
  this file is trying to run, so it's a variant worth naming but not the primary approach here.
- **Recommendation: mean-return sign as the primary rule**, empirical up-frequency as a
  secondary variant to compare against it.

**E. The leakage-critical detail: filtering vs. smoothing.**
A regime model can estimate, for any day *t*, "what state was the market in on day *t*"
using two different kinds of inference: **filtering** (using only data through day *t*,
causal) or **smoothing** (using the entire sequence including future days, non-causal).
Every prediction in the walk-forward **must use filtered, not smoothed, state
probabilities** -- smoothing at test time would leak future price action into "today's"
regime call, silently reintroducing the exact class of leakage this whole session has been
built around eliminating. `statsmodels`' `.filtered_marginal_probabilities` is the causal
one; `.smoothed_marginal_probabilities` is not test-time-safe and is for
in-sample/historical interpretation only (used later in section on regime interpretation,
never for generating a prediction).

In [3]:
# Mechanics demo for Option A (statsmodels MarkovRegression) -- NOT the leakage-safe
# pipeline. Fit on the *full* return history just to see the API and what a fitted model
# looks like before wiring it into the per-fold walk-forward in sections 4-6. Real
# predictions later will only ever use train-fold data at fit time.

demo_returns = raw["close"].pct_change().dropna()
demo_returns.index = raw["date"].iloc[1:].reset_index(drop=True)

# k_regimes=2: two hidden states. trend="c" lets each regime have its own mean (constant)
# return. switching_variance=True lets each regime have its own volatility too -- financial
# regimes typically differ in both drift and vol (e.g. calm-up vs. volatile-down), so leaving
# variance fixed across regimes would force the model to only distinguish states by mean.
demo_model = MarkovRegression(demo_returns, k_regimes=2, trend="c", switching_variance=True)
demo_result = demo_model.fit()

print(demo_result.summary())


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


                        Markov Switching Model Results                        
Dep. Variable:                  close   No. Observations:                 4310
Model:               MarkovRegression   Log Likelihood                9057.274
Date:                Tue, 14 Jul 2026   AIC                         -18102.549
Time:                        12:54:47   BIC                         -18064.337
Sample:                    09-18-2014   HQIC                        -18089.055
                         - 07-06-2026                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0014      0.000      3.508      0.000       0.001       0.002
sigma2         0.0003   1.89e-05     15.408      0.0

## 3. Observation Series Construction

**Scope note vs. §2B:** `MarkovRegression` already lets each regime have its own variance
via `switching_variance=True` (used in the §2 demo) -- regimes differ by *both* drift and
choppiness from the return series alone, without needing volatility fed in as a second,
separate observation. A true joint bivariate emission (return *and* realized volatility
observed simultaneously) would need `hmmlearn`'s multivariate Gaussian HMM instead, which
is a bigger detour (new dependency, different API, its own leakage-safety review) -- noted
as a follow-up variant, not built here. We still compute a realized-volatility series below,
but only as a diagnostic in §9 (does the model's inferred regime volatility track realized
volatility), not as a second model input.

Both series are backward-looking only: `return` is `close.pct_change()`, `realized_vol_14d`
is a trailing 14-day rolling standard deviation of that return -- no centered windows, no
forward information, same discipline as every other feature this session.

In [4]:
model_frame = raw[["date", "close"]].copy()
model_frame["return"] = model_frame["close"].pct_change()
model_frame["realized_vol_14d"] = model_frame["return"].rolling(14).std()

HORIZON_DAYS = 1  # matches the original brief: predict tomorrow's direction
model_frame["target_bullish_tomorrow"] = (model_frame["close"].shift(-HORIZON_DAYS) > model_frame["close"]).astype("float")
model_frame = model_frame.iloc[:-HORIZON_DAYS]  # final row's target is unknowable

model_frame = model_frame.rename(columns={"close": "btc_close"})
model_frame = model_frame.dropna().sort_values("date").reset_index(drop=True)
model_frame["target_bullish_tomorrow"] = model_frame["target_bullish_tomorrow"].astype(int)

model_frame.to_csv(MK4_DIR / "regime_model_frame.csv", index=False)
print(f"Model frame: {model_frame.shape}, {model_frame['date'].min().date()} to {model_frame['date'].max().date()}")
print(f"Up-rate: {model_frame['target_bullish_tomorrow'].mean():.4f}")
model_frame.tail()


Model frame: (4296, 5), 2014-10-01 to 2026-07-05
Up-rate: 0.5247


,date,btc_close,return,realized_vol_14d,target_bullish_tomorrow
4291,2026-07-01,60003.757812,0.024674,0.017192,1
4292,2026-07-02,61485.300781,0.024691,0.017980,1
4293,2026-07-03,62544.199219,0.017222,0.018421,1
4294,2026-07-04,63088.304688,0.008700,0.018316,1
4295,2026-07-05,63547.882812,0.007285,0.017943,1


## 4. Walk-Forward Fold Setup

Same 8-fold expanding `TimeSeriesSplit` and purge as every other notebook this session. The
purge rule is identical in spirit even though there's no "label" being fit here in the
supervised sense -- the boundary between what the regime model is allowed to have seen and
the first test-set prediction still needs to respect the target horizon.

In [5]:
N_SPLITS = 8

dates_all = model_frame["date"].to_numpy()
returns_all = model_frame["return"].to_numpy()
close_all = model_frame["btc_close"].to_numpy()
y_all = model_frame["target_bullish_tomorrow"].to_numpy()

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
fold_indices = []
for train_idx_raw, test_idx in tscv.split(model_frame):
    # Purge: drop training rows whose label (HORIZON_DAYS ahead) reaches into the test period.
    purge_mask = (train_idx_raw + HORIZON_DAYS) >= test_idx.min()
    train_idx = train_idx_raw[~purge_mask]
    n_purged = len(train_idx_raw) - len(train_idx)
    fold_indices.append((train_idx, test_idx, n_purged))

for i, (train_idx, test_idx, n_purged) in enumerate(fold_indices, 1):
    print(f"Fold {i}: train={len(train_idx)} (purged {n_purged})  test={len(test_idx)}  "
          f"({pd.Timestamp(dates_all[test_idx[0]]).date()} to {pd.Timestamp(dates_all[test_idx[-1]]).date()})")


Fold 1: train=479 (purged 1)  test=477  (2016-01-24 to 2017-05-14)
Fold 2: train=956 (purged 1)  test=477  (2017-05-15 to 2018-09-03)
Fold 3: train=1433 (purged 1)  test=477  (2018-09-04 to 2019-12-24)
Fold 4: train=1910 (purged 1)  test=477  (2019-12-25 to 2021-04-14)
Fold 5: train=2387 (purged 1)  test=477  (2021-04-15 to 2022-08-04)
Fold 6: train=2864 (purged 1)  test=477  (2022-08-05 to 2023-11-24)
Fold 7: train=3341 (purged 1)  test=477  (2023-11-25 to 2025-03-15)
Fold 8: train=3818 (purged 1)  test=477  (2025-03-16 to 2026-07-05)


## 5. Fit Regime Model (Train-Only, Per Fold)

`MarkovRegression`/`MarkovAutoregression` fit via EM on each fold's training window only --
refit fresh every fold, never fit once on the full history, consistent with the per-fold
feature selection validated earlier this session. Both the 2-state and 3-state variants (§2C)
get fit here, as independent model variants evaluated side by side.

In [6]:
K_REGIME_VARIANTS = [2, 3]


def fit_regime_model(train_returns, k_regimes):
    """Fit on training-fold returns only. No test data touches this call."""
    model = MarkovRegression(train_returns, k_regimes=k_regimes, trend="c", switching_variance=True)
    return model.fit(disp=False)


print("fit_regime_model defined")


fit_regime_model defined


## 6. Regime Inference and Direction Mapping

For each test-day prediction: infer today's regime using **filtered** (causal) probabilities
only (§2E), then apply the mean-return-sign rule (§2D, primary) and the empirical
up-frequency rule (secondary variant) to produce a bullish/bearish call.

In [7]:
def regime_mean_signs(result, k_regimes):
    """Regime k's fitted mean return, and whether it's positive (§2D primary rule)."""
    means = np.array([result.params[f"const[{k}]"] for k in range(k_regimes)])
    return (means > 0).astype(float), means


def regime_up_frequencies(filtered_probs, y_train, k_regimes):
    """Regime k's probability-weighted empirical next-day up-rate (§2D secondary rule)."""
    freqs = np.zeros(k_regimes)
    for k in range(k_regimes):
        w = filtered_probs.iloc[:, k].to_numpy()
        freqs[k] = np.sum(w * y_train) / np.sum(w)
    return freqs


RULE_NAMES = {"mean_sign": "Mean-sign", "up_freq": "Up-frequency"}
daily_by_variant = {(k, rule): [] for k in K_REGIME_VARIANTS for rule in RULE_NAMES}
regime_fit_log = []

for fold_num, (train_idx, test_idx, n_purged) in enumerate(fold_indices, 1):
    train_returns = pd.Series(returns_all[train_idx], index=pd.to_datetime(dates_all[train_idx]))
    y_train = y_all[train_idx]
    y_test = y_all[test_idx]
    dates_test = dates_all[test_idx]

    # Train+test returns as one array so the causal filter can run across the test period --
    # this is safe (not a leak) because a filtered probability at index t is, by
    # construction, a function only of observations up to and including t. Extending the
    # array past t doesn't change the filtered value AT t. Only .smoothed_* would leak here.
    combined_idx = np.concatenate([train_idx, test_idx])
    combined_returns = pd.Series(returns_all[combined_idx], index=pd.to_datetime(dates_all[combined_idx]))
    test_positions = np.arange(len(train_idx), len(combined_idx))

    for k_regimes in K_REGIME_VARIANTS:
        try:
            train_result = fit_regime_model(train_returns, k_regimes)
        except Exception as e:
            print(f"  Fold {fold_num} k={k_regimes}: fit failed ({e!r}), skipping this fold/variant")
            continue

        mean_sign, mean_vals = regime_mean_signs(train_result, k_regimes)
        up_freq = regime_up_frequencies(train_result.filtered_marginal_probabilities, y_train, k_regimes)
        transition_matrix = train_result.regime_transition[:, :, 0].T  # [from, to], see the check in the chat

        regime_fit_log.append({
            "fold": fold_num, "k_regimes": k_regimes,
            "regime_means": mean_vals.tolist(), "regime_up_freq": up_freq.tolist(),
            "transition_matrix": transition_matrix.tolist(),
        })

        # Filter (NOT fit) the combined train+test sequence using the fixed training-fold
        # parameters -- no re-estimation, so test data never influences the parameters.
        combined_model = MarkovRegression(combined_returns, k_regimes=k_regimes, trend="c", switching_variance=True)
        filtered_result = combined_model.filter(train_result.params)
        test_filtered = filtered_result.filtered_marginal_probabilities.iloc[test_positions].to_numpy()

        proba_up = {
            "mean_sign": test_filtered @ mean_sign,
            "up_freq": test_filtered @ up_freq,
        }

        for i in range(len(test_idx)):
            for rule, proba_arr in proba_up.items():
                daily_by_variant[(k_regimes, rule)].append({
                    "date": pd.Timestamp(dates_test[i]), "fold": fold_num, "actual": int(y_test[i]),
                    "proba_up": float(proba_arr[i]), "pred": int(proba_arr[i] >= 0.5),
                    "always_up_pred": 1,
                })

    print(f"Fold {fold_num}: train={len(train_idx)} (purged {n_purged})  test={len(test_idx)}  "
          f"({pd.Timestamp(dates_test[0]).date()} to {pd.Timestamp(dates_test[-1]).date()})")

daily_frames = {}
for key, rows in daily_by_variant.items():
    k_regimes, rule = key
    df = pd.DataFrame(rows)
    daily_frames[key] = df
    df.to_csv(MK4_DIR / f"regime_daily_predictions_k{k_regimes}_{rule}.csv", index=False)

regime_fit_df = pd.DataFrame(regime_fit_log)
regime_fit_df.to_csv(MK4_DIR / "regime_fit_log.csv", index=False)
print(f"\nDone. {len(daily_frames)} variants, {len(daily_frames[(2, 'mean_sign')])} pooled predictions each.")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inf

Fold 1: train=479 (purged 1)  test=477  (2016-01-24 to 2017-05-14)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


Fold 2: train=956 (purged 1)  test=477  (2017-05-15 to 2018-09-03)


...

Fold 7: train=3341 (purged 1)  test=477  (2023-11-25 to 2025-03-15)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Fold 8: train=3818 (purged 1)  test=477  (2025-03-16 to 2026-07-05)

Done. 4 variants, 3816 pooled predictions each.


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


## 7. Baseline

Always-up only, computed on the same test rows as every regime-model variant, matching every
other notebook this session.

In [8]:
# Always-up is already embedded per-row in daily_by_variant ("always_up_pred": 1). Since
# every variant is evaluated on the identical fold/test structure, the baseline accuracy is
# the same number regardless of which variant's table it's read from -- confirm that here.
baseline_accs = {key: (df["always_up_pred"] == df["actual"]).mean() for key, df in daily_frames.items()}
print("Always-up baseline accuracy (should match across all variants):")
for key, acc in baseline_accs.items():
    print(f"  k={key[0]} {RULE_NAMES[key[1]]:14s} {acc:.4f}")


Always-up baseline accuracy (should match across all variants):
  k=2 Mean-sign      0.5265
  k=2 Up-frequency   0.5265
  k=3 Mean-sign      0.5265
  k=3 Up-frequency   0.5265


## 8. Significance Testing

McNemar's test + fold-clustered bootstrap, identical methodology to every other notebook
this session, run once per regime-model variant (2-state / 3-state x mean-sign /
up-frequency) so the comparison table shows all variants side by side.

In [9]:
def mcnemar(pred_col, actual_col, base_col, df):
    model_correct = (df[pred_col] == df[actual_col]).to_numpy()
    base_correct = (df[base_col] == df[actual_col]).to_numpy()
    b = int(np.sum(model_correct & ~base_correct))
    c = int(np.sum(~model_correct & base_correct))
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else float("nan")
    return b, c, p


def fold_cluster_bootstrap(pred_col, actual_col, base_col, df, n_boot=10000, seed=42):
    d = df.copy()
    d["model_correct"] = (d[pred_col] == d[actual_col]).astype(float)
    d["base_correct"] = (d[base_col] == d[actual_col]).astype(float)
    fold_stats = d.groupby("fold").agg(n=("model_correct", "size"),
                                        model_acc=("model_correct", "mean"),
                                        base_acc=("base_correct", "mean")).reset_index()
    n_folds = len(fold_stats)
    fold_ns = fold_stats["n"].to_numpy()
    fold_model_sum = (fold_stats["model_acc"] * fold_stats["n"]).to_numpy()
    fold_base_sum = (fold_stats["base_acc"] * fold_stats["n"]).to_numpy()
    rng = np.random.default_rng(seed)
    boot_edges = np.empty(n_boot)
    for i in range(n_boot):
        picks = rng.integers(0, n_folds, size=n_folds)
        total_n = fold_ns[picks].sum()
        boot_edges[i] = fold_model_sum[picks].sum() / total_n - fold_base_sum[picks].sum() / total_n
    lo, hi = np.percentile(boot_edges, [2.5, 97.5])
    observed = fold_model_sum.sum() / fold_ns.sum() - fold_base_sum.sum() / fold_ns.sum()
    return n_folds, observed, lo, hi


summary_rows = []
for (k_regimes, rule), df in daily_frames.items():
    model_acc = (df["pred"] == df["actual"]).mean()
    base_acc = (df["always_up_pred"] == df["actual"]).mean()
    b, c, p = mcnemar("pred", "actual", "always_up_pred", df)
    n_folds, observed, lo, hi = fold_cluster_bootstrap("pred", "actual", "always_up_pred", df)
    summary_rows.append({
        "k_regimes": k_regimes, "rule": RULE_NAMES[rule],
        "model_acc": model_acc, "baseline_acc": base_acc,
        "edge": observed, "mcnemar_p": p,
        "bootstrap_ci_lo": lo, "bootstrap_ci_hi": hi,
        "ci_excludes_zero": not (lo <= 0 <= hi),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(MK4_DIR / "regime_significance_summary.csv", index=False)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
summary


,k_regimes,rule,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero
0,2,Mean-sign,0.5238,0.5265,-0.0026,0.2451,-0.0079,0.0000,False
1,2,Up-frequency,0.5265,0.5265,0.0000,NaN,0.0000,0.0000,False
2,3,Mean-sign,0.5063,0.5265,-0.0202,0.0002,-0.0527,0.0000,False
3,3,Up-frequency,0.5246,0.5265,-0.0018,0.3240,-0.0055,0.0000,False


## 9. Which Regimes Actually Got Identified

Interpretation pass using **smoothed** (non-causal, in-sample only) probabilities -- safe to
use here because this section characterizes what the model found historically, it never
feeds a prediction. Look at regime persistence (how many days a regime typically lasts),
each regime's mean return/volatility, and whether the identified regimes line up with
recognizable historical periods (2018 bear, 2020 COVID crash, 2021 bull, 2022 bear, etc.) --
a sanity check on whether the model is finding real structure before trusting any accuracy
number from section 8.

In [10]:
full_returns = pd.Series(returns_all, index=pd.to_datetime(dates_all))

interp_results = {}
for k_regimes in K_REGIME_VARIANTS:
    interp_results[k_regimes] = fit_regime_model(full_returns, k_regimes)

for k_regimes, result in interp_results.items():
    print(f"{'=' * 60}\nk = {k_regimes} regimes -- fit on full history (in-sample, interpretation only)\n{'=' * 60}")

    # --- The actual fitted Markov transition matrix, formatted [from, to] ---
    trans = result.regime_transition[:, :, 0].T
    trans_df = pd.DataFrame(trans,
                             index=[f"from regime {i}" for i in range(k_regimes)],
                             columns=[f"to regime {j}" for j in range(k_regimes)])
    print("\nTransition matrix P(row -> column):")
    print(trans_df.round(4))

    # --- Expected regime duration: 1 / (1 - P(stay)) ---
    durations = 1 / (1 - np.diag(trans))
    print("\nExpected duration once in each regime (days):")
    for k in range(k_regimes):
        print(f"  regime {k}: {durations[k]:6.1f} days")

    # --- Mean return / implied volatility per regime ---
    _, mean_vals = regime_mean_signs(result, k_regimes)
    sigma2_vals = np.array([result.params[f"sigma2[{k}]"] for k in range(k_regimes)])
    print("\nRegime mean daily return / implied daily volatility:")
    for k in range(k_regimes):
        print(f"  regime {k}: mean={mean_vals[k]:+.4%}   vol={np.sqrt(sigma2_vals[k]):.4%}")
    print()


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


k = 2 regimes -- fit on full history (in-sample, interpretation only)

Transition matrix P(row -> column):
               to regime 0  to regime 1
from regime 0       0.9039       0.0961
from regime 1       0.1708       0.8292

Expected duration once in each regime (days):
  regime 0:   10.4 days
  regime 1:    5.9 days

Regime mean daily return / implied daily volatility:
  regime 0: mean=+0.1365%   vol=1.7062%
  regime 1: mean=+0.2567%   vol=5.3311%

k = 3 regimes -- fit on full history (in-sample, interpretation only)

Transition matrix P(row -> column):
               to regime 0  to regime 1  to regime 2
from regime 0       0.6564       0.3271       0.0164
from regime 1       0.2499       0.7300       0.0201
from regime 2       0.0000       0.0687       0.9312

Expected duration once in each regime (days):
  regime 0:    2.9 days
  regime 1:    3.7 days
  regime 2:   14.5 days

Regime mean daily return / implied daily volatility:
  regime 0: mean=+0.0546%   vol=0.9552%
  regime 1:

In [11]:
# Historical alignment check -- smoothed (non-causal) probabilities, in-sample only, safe
# here because this never feeds a prediction. Group the smoothed hard-assignment into
# contiguous blocks and show the longest ones: do they land on recognizable periods
# (2018 bear, 2020 COVID crash, 2021 bull, 2022 bear)?
for k_regimes, result in interp_results.items():
    smoothed = result.smoothed_marginal_probabilities
    hard_state = smoothed.to_numpy().argmax(axis=1)
    dates = model_frame["date"].to_numpy()

    blocks = []
    start = 0
    for i in range(1, len(hard_state) + 1):
        if i == len(hard_state) or hard_state[i] != hard_state[start]:
            blocks.append((hard_state[start], dates[start], dates[i - 1], i - start))
            start = i

    blocks_df = pd.DataFrame(blocks, columns=["regime", "start", "end", "n_days"])
    blocks_df = blocks_df.sort_values("n_days", ascending=False)

    print(f"k = {k_regimes}: top 8 longest regime blocks (smoothed, in-sample)")
    for _, row in blocks_df.head(8).iterrows():
        print(f"  regime {row['regime']}: {pd.Timestamp(row['start']).date()} to "
              f"{pd.Timestamp(row['end']).date()}  ({row['n_days']} days)")
    print()


k = 2: top 8 longest regime blocks (smoothed, in-sample)
  regime 0: 2025-05-09 to 2025-10-09  (154 days)
  regime 0: 2016-02-05 to 2016-05-26  (112 days)
  regime 0: 2016-08-04 to 2016-10-31  (89 days)
  regime 0: 2024-08-28 to 2024-11-05  (70 days)
  regime 0: 2018-09-08 to 2018-11-13  (67 days)
  regime 1: 2017-11-23 to 2018-01-24  (63 days)
  regime 0: 2015-08-26 to 2015-10-27  (63 days)
  regime 0: 2022-11-12 to 2023-01-10  (60 days)

k = 3: top 8 longest regime blocks (smoothed, in-sample)
  regime 2: 2017-10-27 to 2018-04-27  (183 days)
  regime 2: 2020-12-16 to 2021-03-06  (81 days)
  regime 2: 2021-04-19 to 2021-06-29  (72 days)
  regime 2: 2018-11-14 to 2018-12-29  (46 days)
  regime 2: 2017-05-08 to 2017-06-18  (42 days)
  regime 2: 2017-07-10 to 2017-08-16  (38 days)
  regime 2: 2015-01-02 to 2015-02-04  (34 days)
  regime 1: 2021-07-19 to 2021-08-20  (33 days)



## 10. Summary

**None of the four variants beat Always-up, and one root cause explains all of it.**

| Variant | Accuracy | vs. Always-up (52.65%) |
|---|---|---|
| 2-state, mean-sign | 52.4% | edge -0.26pp, not significant |
| 2-state, up-frequency | 52.65% | **identical to Always-up** -- 0 of 3,816 predictions disagreed |
| 3-state, mean-sign | 50.6% | edge -2.02pp, McNemar p=0.0002 but bootstrap CI touches zero at the boundary -- not significant by our stricter rule, but the worst-performing variant by a clear margin |
| 3-state, up-frequency | 52.5% | edge -0.18pp, not significant |

**Why: this data essentially never contains a bearish regime.** Pulling the actual fitted
regime means from `regime_fit_log.csv` (not the summary table -- the raw numbers behind
it): in **6 of 8 folds at k=2 and 6 of 8 folds at k=3, every single regime the model found
has a positive mean daily return.** Only the two earliest folds -- whose training windows
include the 2018 crash -- found any regime with a negative mean, and even those aren't
present once the training window extends further into the bull-dominated years that
followed. A return-only regime model over this particular BTC history doesn't find "bull vs.
bear," it finds **"calm growth" vs. "volatile growth."** That's exactly why 2-state
up-frequency collapsed to literally Always-up: if every regime's next-day up-rate is above
50%, the weighted-average prediction is "up" regardless of which regime the filter says
we're in, every single day.

**The model isn't broken -- it's finding real structure, just not directional structure.**
§9's historical alignment check is the evidence: the high-volatility regime's longest blocks
land squarely on recognizable turbulent stretches (Oct 2017-Apr 2018 -- the top and crash;
Dec 2020-Mar 2021 and Apr-Jun 2021 -- the 2020-21 bull run's volatile legs). The transition
matrix (§9, k=2) shows real persistence too -- regime 0 averages ~10 days before switching,
regime 1 ~6 days -- not random noise. The regime-switching mechanism is doing its job
correctly; the mismatch is that **regime membership here tracks volatility, and volatility
doesn't reliably predict direction on a market that trends up through both its calm and
turbulent stretches.**

**What this adds to the session's evidence, and what it doesn't.** This is a genuinely
different kind of negative result from the five classifiers tested earlier -- not "no
signal in the features," but "the *process itself*, described the simplest way (return-only,
switching mean and variance), doesn't have a bearish state to find in this window." It
doesn't rule out regime-switching as an approach -- see the options below -- it rules out
*this* specific formulation of it.

**Natural next steps, not built here:**
- **Exogenous regressors** -- `MarkovRegression` supports switching or non-switching `exog`
  (e.g. on-chain exchange flow, macro), which could let regimes separate on more than just
  return magnitude, closer to §2B's richer multivariate goal without needing `hmmlearn`.
- **Excess return instead of raw return** -- de-mean by a slow trailing average before
  fitting, so regimes aren't dominated by the drift term and have more room to represent
  direction independent of the market's overall trend.
- **A 30-day horizon**, matching the one place this whole session found a (later
  non-replicated) significant result -- regimes persisting ~6-15 days per §9 are a poor
  match for a 1-day-ahead target but a closer match to a multi-week one.

## 11. Shared Walk-Forward Helper for the Follow-Up Experiments

The three follow-ups from §10 (excess return, exogenous regressor, 30-day horizon) all need
the same purged 8-fold walk-forward, causal-filter, dual-rule prediction machinery as
§4-6 -- just with a different observation series, target, or an added exogenous term. Rather
than copy the ~80-line loop three times, this generalizes it into one function, parameterized
by which columns to use. Everything else -- the purge logic, the filtered-not-smoothed rule,
the mean-sign/up-frequency dual prediction -- is unchanged from §4-6.

In [12]:
def run_regime_walkforward(frame, endog_col, target_col, horizon_days, k_list,
                            exog_col=None, n_splits=8, verbose=True):
    """Purged 8-fold walk-forward for a MarkovRegression variant.

    frame must be pre-sorted by date with no gaps relevant to endog_col/target_col/exog_col.
    Returns {(k_regimes, rule): daily_predictions_df}.
    """
    dates = frame["date"].to_numpy()
    endog_all = frame[endog_col].to_numpy()
    y_all_local = frame[target_col].to_numpy()
    exog_all = frame[exog_col].to_numpy() if exog_col else None

    tscv = TimeSeriesSplit(n_splits=n_splits)
    daily = {(k, rule): [] for k in k_list for rule in RULE_NAMES}
    fold_num = 0

    for train_idx_raw, test_idx in tscv.split(frame):
        fold_num += 1
        purge_mask = (train_idx_raw + horizon_days) >= test_idx.min()
        train_idx = train_idx_raw[~purge_mask]
        n_purged = len(train_idx_raw) - len(train_idx)

        y_train = y_all_local[train_idx]
        y_test = y_all_local[test_idx]
        dates_test = dates[test_idx]
        combined_idx = np.concatenate([train_idx, test_idx])
        test_positions = np.arange(len(train_idx), len(combined_idx))

        for k in k_list:
            fit_kwargs = {}
            if exog_col:
                fit_kwargs = {"exog": exog_all[train_idx], "switching_exog": True}

            train_series = pd.Series(endog_all[train_idx], index=pd.to_datetime(dates[train_idx]))
            try:
                model = MarkovRegression(train_series, k_regimes=k, trend="c",
                                          switching_variance=True, **fit_kwargs)
                result = model.fit(disp=False)
            except Exception as e:
                if verbose:
                    print(f"  Fold {fold_num} k={k}: fit failed ({e!r}), skipping")
                continue

            combined_kwargs = {}
            if exog_col:
                combined_kwargs = {"exog": exog_all[combined_idx], "switching_exog": True}
            combined_series = pd.Series(endog_all[combined_idx], index=pd.to_datetime(dates[combined_idx]))
            combined_model = MarkovRegression(combined_series, k_regimes=k, trend="c",
                                               switching_variance=True, **combined_kwargs)
            filtered_result = combined_model.filter(result.params)
            test_filtered = filtered_result.filtered_marginal_probabilities.iloc[test_positions].to_numpy()

            up_freq = regime_up_frequencies(result.filtered_marginal_probabilities, y_train, k)
            proba_up_freq = test_filtered @ up_freq

            if exog_col:
                # Predicted mean_t = const[k] + beta[k]*exog_t -- varies day to day within a
                # regime, unlike the fixed regime mean used elsewhere. Mixed across regimes
                # by the (causal) filtered probability, same as every other rule here.
                const_vals = np.array([result.params[f"const[{kk}]"] for kk in range(k)])
                beta_vals = np.array([result.params[f"x1[{kk}]"] for kk in range(k)])
                exog_test = exog_all[test_idx]
                predicted_mean = const_vals[None, :] + beta_vals[None, :] * exog_test[:, None]
                proba_mean_sign = np.sum(test_filtered * (predicted_mean > 0), axis=1)
            else:
                mean_sign, _ = regime_mean_signs(result, k)
                proba_mean_sign = test_filtered @ mean_sign

            for i in range(len(test_idx)):
                for rule, proba in [("mean_sign", proba_mean_sign[i]), ("up_freq", proba_up_freq[i])]:
                    daily[(k, rule)].append({
                        "date": pd.Timestamp(dates_test[i]), "fold": fold_num, "actual": int(y_test[i]),
                        "proba_up": float(proba), "pred": int(proba >= 0.5), "always_up_pred": 1,
                    })

        if verbose:
            print(f"  Fold {fold_num}: train={len(train_idx)} (purged {n_purged})  test={len(test_idx)}  "
                  f"({pd.Timestamp(dates_test[0]).date()} to {pd.Timestamp(dates_test[-1]).date()})")

    return {key: pd.DataFrame(rows) for key, rows in daily.items()}


def summarize_variants(daily_frames_local, label):
    rows = []
    for (k, rule), df in daily_frames_local.items():
        if len(df) == 0:
            continue
        model_acc = (df["pred"] == df["actual"]).mean()
        base_acc = (df["always_up_pred"] == df["actual"]).mean()
        b, c, p = mcnemar("pred", "actual", "always_up_pred", df)
        n_folds, observed, lo, hi = fold_cluster_bootstrap("pred", "actual", "always_up_pred", df)
        rows.append({
            "experiment": label, "k_regimes": k, "rule": RULE_NAMES[rule],
            "model_acc": model_acc, "baseline_acc": base_acc, "edge": observed,
            "mcnemar_p": p, "bootstrap_ci_lo": lo, "bootstrap_ci_hi": hi,
            "ci_excludes_zero": not (lo <= 0 <= hi),
        })
    return pd.DataFrame(rows)


print("run_regime_walkforward and summarize_variants defined")


run_regime_walkforward and summarize_variants defined


## 12. Follow-Up A -- Excess Return (De-Meaned)

§10's diagnosis: raw daily return over this window has such a strong positive drift that
almost every regime the model finds ends up with a positive mean, leaving no genuine
"bearish" state to predict from. Excess return subtracts a slow trailing average of *prior*
returns (60-day, using `shift(1)` before the rolling window so today's own return never
contributes to today's baseline) before fitting -- regimes now describe departures from the
recent trend rather than the trend itself, which gives a negative-mean regime room to exist
even in an overall uptrend.

In [13]:
TREND_WINDOW = 60

frame_excess = model_frame.copy()
frame_excess["trend"] = frame_excess["return"].shift(1).rolling(TREND_WINDOW).mean()
frame_excess["excess_return"] = frame_excess["return"] - frame_excess["trend"]
frame_excess = frame_excess.dropna().sort_values("date").reset_index(drop=True)

print(f"Excess-return frame: {frame_excess.shape}, {frame_excess['date'].min().date()} to {frame_excess['date'].max().date()}")
print(f"Mean raw return:    {frame_excess['return'].mean():+.4%}")
print(f"Mean excess return: {frame_excess['excess_return'].mean():+.4%}")

daily_excess = run_regime_walkforward(frame_excess, endog_col="excess_return",
                                       target_col="target_bullish_tomorrow",
                                       horizon_days=HORIZON_DAYS, k_list=K_REGIME_VARIANTS)

for key, df in daily_excess.items():
    k, rule = key
    df.to_csv(MK4_DIR / f"regime_excess_daily_predictions_k{k}_{rule}.csv", index=False)

summary_excess = summarize_variants(daily_excess, "Excess return (1d)")
summary_excess.to_csv(MK4_DIR / "regime_excess_significance_summary.csv", index=False)
summary_excess


Excess-return frame: (4236, 7), 2014-11-30 to 2026-07-05
Mean raw return:    +0.1822%
Mean excess return: -0.0029%


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency inform

  Fold 1: train=475 (purged 1)  test=470  (2016-03-20 to 2017-07-02)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


...

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


  Fold 8: train=3765 (purged 1)  test=470  (2025-03-23 to 2026-07-05)


,experiment,k_regimes,rule,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero
0,Excess return (1d),2,Mean-sign,0.4981,0.5255,-0.0274,0.0044,-0.0479,-0.0069,True
1,Excess return (1d),2,Up-frequency,0.5255,0.5255,0.0000,NaN,0.0000,0.0000,False
2,Excess return (1d),3,Mean-sign,0.5061,0.5255,-0.0194,0.0173,-0.0285,-0.0098,True
3,Excess return (1d),3,Up-frequency,0.5255,0.5255,0.0000,NaN,0.0000,0.0000,False


## 13. Follow-Up B -- Regime Model with an Exogenous On-Chain Regressor

`switching_exog=True` gives each regime its own coefficient on an external variable, so the
predicted mean for day *t* becomes `const[k] + beta[k] * exog_t` instead of a fixed
per-regime constant -- letting the model use information beyond the return series' own
history without needing `hmmlearn`. Regressor: **exchange netflow**, 30-day rolling z-score
(same-day value -- on-chain data is realized by end of day, no publication lag needed, same
treatment as on-chain features elsewhere this session). Net inflow to exchanges is a classic
directional signal (coins moving to exchanges is often read as preparation to sell), which
raw/excess return alone can't see.

In [14]:
frame_exog = model_frame.merge(raw[["date", "exchange_netflow_btc"]], on="date", how="left")
frame_exog["netflow_z"] = (
    (frame_exog["exchange_netflow_btc"] - frame_exog["exchange_netflow_btc"].rolling(30).mean())
    / frame_exog["exchange_netflow_btc"].rolling(30).std()
)
frame_exog = frame_exog.dropna().sort_values("date").reset_index(drop=True)

print(f"Exog frame: {frame_exog.shape}, {frame_exog['date'].min().date()} to {frame_exog['date'].max().date()}")

daily_exog = run_regime_walkforward(frame_exog, endog_col="return",
                                     target_col="target_bullish_tomorrow",
                                     horizon_days=HORIZON_DAYS, k_list=K_REGIME_VARIANTS,
                                     exog_col="netflow_z")

for key, df in daily_exog.items():
    k, rule = key
    df.to_csv(MK4_DIR / f"regime_exog_daily_predictions_k{k}_{rule}.csv", index=False)

summary_exog = summarize_variants(daily_exog, "Exog netflow (1d)")
summary_exog.to_csv(MK4_DIR / "regime_exog_significance_summary.csv", index=False)
summary_exog


Exog frame: (4267, 7), 2014-10-30 to 2026-07-05


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


  Fold 1: train=474 (purged 1)  test=474  (2016-02-17 to 2017-06-04)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


...

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


  Fold 8: train=3792 (purged 1)  test=474  (2025-03-19 to 2026-07-05)


,experiment,k_regimes,rule,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero
0,Exog netflow (1d),2,Mean-sign,0.5214,0.5264,-0.0050,0.3089,-0.0145,0.0032,False
1,Exog netflow (1d),2,Up-frequency,0.5264,0.5264,0.0000,NaN,0.0000,0.0000,False
2,Exog netflow (1d),3,Mean-sign,0.5129,0.5264,-0.0134,0.0296,-0.0195,-0.0076,True
3,Exog netflow (1d),3,Up-frequency,0.5264,0.5264,0.0000,1.0000,0.0000,0.0000,False


## 14. Follow-Up C -- 30-Day Horizon

Same raw-return regime model as §4-9 (no excess-return or exog changes -- isolating the
horizon as the only variable), retargeted to `close.shift(-30) > close`. §9 found regimes
persisting roughly 6-15 days on average -- a poor match for a 1-day-ahead target where most
of a regime's life happens *after* the prediction is scored, but a closer match to a
multi-week target where the regime is more likely to still hold when the outcome is
realized. Purge scales with the horizon exactly as it did in the multi-day notebook: up to
29 training rows dropped per fold.

In [15]:
HORIZON_30D = 30

frame_30d = raw[["date", "close"]].copy()
frame_30d["return"] = frame_30d["close"].pct_change()
frame_30d["target_bullish_30d"] = (frame_30d["close"].shift(-HORIZON_30D) > frame_30d["close"]).astype("float")
frame_30d = frame_30d.iloc[:-HORIZON_30D]
frame_30d = frame_30d.rename(columns={"close": "btc_close"})
frame_30d = frame_30d.dropna().sort_values("date").reset_index(drop=True)
frame_30d["target_bullish_30d"] = frame_30d["target_bullish_30d"].astype(int)

print(f"30-day frame: {frame_30d.shape}, {frame_30d['date'].min().date()} to {frame_30d['date'].max().date()}")
print(f"30-day up-rate: {frame_30d['target_bullish_30d'].mean():.4f}")

daily_30d = run_regime_walkforward(frame_30d, endog_col="return", target_col="target_bullish_30d",
                                    horizon_days=HORIZON_30D, k_list=K_REGIME_VARIANTS)

for key, df in daily_30d.items():
    k, rule = key
    df.to_csv(MK4_DIR / f"regime_30d_daily_predictions_k{k}_{rule}.csv", index=False)

summary_30d = summarize_variants(daily_30d, "30-day horizon")
summary_30d.to_csv(MK4_DIR / "regime_30d_significance_summary.csv", index=False)
summary_30d


30-day frame: (4280, 4), 2014-09-18 to 2026-06-06
30-day up-rate: 0.5617


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency inform

  Fold 1: train=450 (purged 30)  test=475  (2016-01-11 to 2017-04-29)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inf

...

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


  Fold 8: train=3775 (purged 30)  test=475  (2025-02-17 to 2026-06-06)


,experiment,k_regimes,rule,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero
0,30-day horizon,2,Mean-sign,0.5600,0.5671,-0.0071,0.0024,-0.0213,0.0000,False
1,30-day horizon,2,Up-frequency,0.5626,0.5671,-0.0045,0.0115,-0.0134,0.0000,False
2,30-day horizon,3,Mean-sign,0.5618,0.5671,-0.0053,0.1657,-0.0126,0.0000,False
3,30-day horizon,3,Up-frequency,0.5597,0.5671,-0.0074,0.0020,-0.0221,0.0000,False


## 15. Combined Comparison

In [16]:
baseline_summary = summary.copy()
baseline_summary.insert(0, "experiment", "Raw return (1d)")

combined = pd.concat([baseline_summary, summary_excess, summary_exog, summary_30d], ignore_index=True)
combined.to_csv(MK4_DIR / "regime_all_experiments_summary.csv", index=False)
combined


,experiment,k_regimes,rule,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero
0,Raw return (1d),2,Mean-sign,0.5238,0.5265,-0.0026,0.2451,-0.0079,0.0000,False
1,Raw return (1d),2,Up-frequency,0.5265,0.5265,0.0000,NaN,0.0000,0.0000,False
2,Raw return (1d),3,Mean-sign,0.5063,0.5265,-0.0202,0.0002,-0.0527,0.0000,False
3,Raw return (1d),3,Up-frequency,0.5246,0.5265,-0.0018,0.3240,-0.0055,0.0000,False
4,Excess return (1d),2,Mean-sign,0.4981,0.5255,-0.0274,0.0044,-0.0479,-0.0069,True
5,Excess return (1d),2,Up-frequency,0.5255,0.5255,0.0000,NaN,0.0000,0.0000,False
6,Excess return (1d),3,Mean-sign,0.5061,0.5255,-0.0194,0.0173,-0.0285,-0.0098,True
7,Excess return (1d),3,Up-frequency,0.5255,0.5255,0.0000,NaN,0.0000,0.0000,False
8,Exog netflow (1d),2,Mean-sign,0.5214,0.5264,-0.0050,0.3089,-0.0145,0.0032,False
9,Exog netflow (1d),2,Up-frequency,0.5264,0.5264,0.0000,NaN,0.0000,0.0000,False


## 16. Follow-Up D -- Moving-Average Trend Rule

Not a regime model -- a fixed technical rule, tested for comparison: **predict bullish
when today's close is above its trailing simple moving average, bearish when below.** This
is genuinely different from the removed Persistence baseline (Persistence asked "did
yesterday go up," this asks "is price above its own trend") and from the regime models
above (no latent state, no fitted parameters at all -- the moving average is a fixed,
deterministic function of price history). Because it has no parameters estimated from
labels, there's nothing for a training fold to leak into a test fold -- but predictions are
still pooled across the same 8 fold/test splits as every other experiment in this notebook
so the comparison is apples to apples. A broad sweep of window lengths tested side by
side -- 5, 10, 15, 20, 30, 40, 50, 75, 100, 125, 150, 200, 250, and 300 days -- covering
everything from a fast short-term trend filter to a slow long-term one, so this isn't
just a lucky/unlucky pick of one window.

In [17]:
MA_WINDOWS = [5, 10, 15, 20, 30, 40, 50, 75, 100, 125, 150, 200, 250, 300]

ma_daily = {}
for w in MA_WINDOWS:
    ma = model_frame["btc_close"].rolling(w).mean()
    pred_bullish = (model_frame["btc_close"] > ma).astype(int)

    rows = []
    for fold_num, (train_idx, test_idx, n_purged) in enumerate(fold_indices, 1):
        for i in test_idx:
            if pd.isna(ma.iloc[i]):
                continue
            rows.append({
                "date": model_frame["date"].iloc[i], "fold": fold_num,
                "actual": int(y_all[i]), "pred": int(pred_bullish.iloc[i]),
                "always_up_pred": 1,
            })
    ma_daily[w] = pd.DataFrame(rows)
    ma_daily[w].to_csv(MK4_DIR / f"regime_ma_trend_daily_predictions_{w}d.csv", index=False)

ma_rows = []
for w, df in ma_daily.items():
    model_acc = (df["pred"] == df["actual"]).mean()
    base_acc = (df["always_up_pred"] == df["actual"]).mean()
    b, c, p = mcnemar("pred", "actual", "always_up_pred", df)
    n_folds, observed, lo, hi = fold_cluster_bootstrap("pred", "actual", "always_up_pred", df)
    ma_rows.append({
        "experiment": "MA trend rule (1d)", "ma_window": w,
        "model_acc": model_acc, "baseline_acc": base_acc, "edge": observed,
        "mcnemar_p": p, "bootstrap_ci_lo": lo, "bootstrap_ci_hi": hi,
        "ci_excludes_zero": not (lo <= 0 <= hi), "n_predictions": len(df),
    })

summary_ma = pd.DataFrame(ma_rows)
summary_ma.to_csv(MK4_DIR / "regime_ma_trend_significance_summary.csv", index=False)
summary_ma

,experiment,ma_window,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero,n_predictions
0,MA trend rule (1d),5,0.4953,0.5265,-0.0312,0.0045,-0.0487,-0.0181,True,3816
1,MA trend rule (1d),10,0.4969,0.5265,-0.0296,0.0073,-0.0469,-0.0149,True,3816
2,MA trend rule (1d),15,0.5079,0.5265,-0.0186,0.0932,-0.0343,-0.0055,True,3816
3,MA trend rule (1d),20,0.5076,0.5265,-0.0189,0.0876,-0.0367,-0.0026,True,3816
4,MA trend rule (1d),30,0.5113,0.5265,-0.0152,0.1681,-0.0294,-0.0016,True,3816
5,MA trend rule (1d),40,0.5215,0.5265,-0.0050,0.6618,-0.0210,0.0089,False,3816
6,MA trend rule (1d),50,0.5244,0.5265,-0.0021,0.8638,-0.0173,0.0123,False,3816
7,MA trend rule (1d),75,0.5231,0.5265,-0.0034,0.7660,-0.0225,0.0142,False,3816
8,MA trend rule (1d),100,0.5207,0.5265,-0.0058,0.5964,-0.0233,0.0115,False,3816
9,MA trend rule (1d),125,0.5257,0.5265,-0.0008,0.9590,-0.0149,0.0123,False,3816


## 17. Updated Summary

**All three follow-ups ran clean. None of them helped -- two of them made things
significantly worse.**

| Experiment | Best variant | Edge vs. Always-up | Significant? |
|---|---|---|---|
| Raw return (§4-10, baseline) | k=3 mean-sign | -2.02pp | No (CI touches zero) |
| **Excess return (§12)** | k=2 mean-sign | **-2.74pp** | **Yes -- significantly worse** |
| **Exog netflow (§13)** | k=3 mean-sign | **-1.34pp** | **Yes -- significantly worse** |
| 30-day horizon (§14) | k=3 up-freq | -0.74pp | Yes -- significantly worse |

Every single one of the 16 model/rule combinations across all four experiments sits at or
below Always-up. Not one produced a positive edge, let alone a significant one.

**Excess return didn't just fail to help -- it made the mean-sign rule significantly worse
(§12).** The de-trending succeeded on its own terms (it does let negative-mean regimes
exist, unlike raw return), but predicting "down" whenever excess return is in a
below-trend regime is a losing bet on this market: BTC exhibits daily-frequency momentum,
not mean reversion, so a regime defined by "temporarily below its own recent trend" is
more often followed by continuation than reversal. The fix for §10's diagnosis
(no bearish regime exists) inadvertently built a regime that's bearish for the wrong
reason -- it responds to a short-term dip, which this market tends to keep dipping through
or resume trending past, not reverse.

**The exogenous netflow regressor added information the return series doesn't have, and it
still didn't help (§13).** The 3-state mean-sign variant is significantly worse than
Always-up, same pattern as the other two. Net exchange flow may simply not carry the
directional information the classic "coins moving to exchanges predicts selling" narrative
claims, at least not net of what daily price action already captures -- consistent with
every other on-chain feature tested across this session's other notebooks.

**The 30-day horizon didn't fix anything either (§14) -- if anything it's the most
consistently negative result of the four experiments**, with 3 of 4 variants significantly
below Always-up. Longer regime persistence (§9: 6-15 days) didn't translate into a better
match for a 30-day target the way the hypothesis suggested.

**The up-frequency rule's collapse to literal Always-up (§10) generalizes across most of
these experiments too** -- 6 of the 8 new up-frequency variants have an edge of exactly
0.0000 (identical predictions to Always-up on every test day). This is the same mechanism
as before: whenever every regime's historical up-rate exceeds 50%, the probability-weighted
prediction is "up" regardless of which regime the filter reports. It only broke from exact
Always-up at the 30-day horizon, where regime-conditional up-rates evidently do straddle
50% -- but even there, the result was significantly worse, not better.

**Where this leaves the regime-switching approach overall.** Four independent
formulations now -- raw return, excess return, return+exogenous flow, and a 30-day
horizon -- have all failed to find a directional edge, with two producing results
significantly worse than simply staying long. Combined with the five classifiers tested
earlier in this session, this is a second architecturally distinct approach (generative
latent-state models, not discriminative classifiers) reaching the same conclusion: there is
no easily-discoverable directional edge in this project's available price, on-chain, and
macro data at these horizons. The honest next moves are the ones already on the table from
before this file existed -- genuinely new data (funding rates, order flow) or reframing the
question entirely (confidence-thresholded predictions, trading-strategy evaluation with
real costs) -- not another variation on regime-switching.

**A fifth follow-up -- a fixed moving-average trend rule, not a fitted model at all
(§16) -- makes the pattern even starker.** Predicting bullish when price is above its
trailing SMA and bearish when below, swept across 14 window lengths from 5 to 300 days:
**every single one has a negative edge vs. Always-up, and the five shortest (5, 10, 15,
20, 30 days) are significantly worse** (bootstrap CIs entirely negative, e.g. the 5-day
window: edge -3.12pp, CI [-0.0487, -0.0181]). Edges drift toward (but never cross) zero as
the window lengthens -- a 300-day SMA is so slow to flip "below" during this heavily
uptrending window that its predictions increasingly resemble Always-up itself, without ever
actually beating it. This is the clearest illustration yet of why every baseline this
session keeps landing in the same place: on a market that trended up through most of this
period, any rule that ever predicts "down" pays for it, and the only way to avoid that cost
is to predict "up" so consistently that the rule stops being a rule and just becomes
Always-up by another route.

**Five independent formulations now, all pointing the same direction:** raw-return
regimes, excess-return regimes, exogenous-regressor regimes, a 30-day regime horizon, and
a classic 14-window moving-average trend sweep. None beat Always-up; four of the five
produced results significantly worse than it. Combined with the five classifiers from
earlier in this session, that's ten independent modeling attempts across two entirely
different paradigms (discriminative classifiers and generative/rule-based regime and trend
approaches) all reaching the same conclusion.